# 01 Pipeline Walkthrough

## 1. Assignment Requirements

The coursework task is to build a metamodel on top of supplied primary trading signals. The metamodel predicts whether a primary signal should be followed, not raw future return direction.

A training sample is a `date x instrument x non-zero primary signal` trade opportunity. `X` is the primary signal plus lagged OHLCV-derived features and optional regime/history features. `y` is a binary triple-barrier meta-label: `1` if the primary signal hits profit-taking before stop-loss within the vertical barrier, and `0` otherwise. Zero primary signals are not genuine trade opportunities; in the submission file they receive neutral probability `0.5` if required.

Triple-barrier labels are path-aware because they use the first barrier touched through the holding path, not only the final horizon return. Barrier width uses lagged volatility so each instrument's threshold scales to recent conditions without using future information. Features must be lagged so the model does not see same-day or future OHLCV. Model and label selection must use only training-period chronological validation, not hidden test data. The final deliverable is `date,instrument,prediction` with probabilities in `[0,1]`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from coursework.src.config import *

from coursework.src.data_loader import load_ohlcv, load_primary_signals, melt_signals, merge_price_and_signal_data
from coursework.src.validation import validate_input_files, validate_date_instrument_panel, validate_no_duplicates, validate_prediction_file, validate_no_lookahead_features, validate_barrier_end_dates
from coursework.src.features import build_features, add_signal_features, add_controlled_features
from coursework.src.labeling import triple_barrier_labels, label_diagnostics
from coursework.src.models import ExperimentSpec, feature_columns_for_experiment, fit_fixed_family_final_models
from coursework.src.evaluation import final_evaluation_summary
from coursework.src.importance import cluster_permutation_importance
from coursework.src.reporting import make_predictions, make_strategy_weights

## 2. Data Loading

In [ ]:
validate_input_files(DEFAULT_OHLCV_PATH, DEFAULT_PRIMARY_SIGNALS_PATH)
ohlcv = load_ohlcv(DEFAULT_OHLCV_PATH)
signals = load_primary_signals(DEFAULT_PRIMARY_SIGNALS_PATH)
signals_long = melt_signals(signals)
ohlcv.head(), signals.head(), signals_long.head()

## 3. Data Validation

In [ ]:
panel_summary = validate_date_instrument_panel(ohlcv, INSTRUMENTS)
validate_no_duplicates(signals_long)
panel_summary

## 4. Feature Construction

In [ ]:
features = build_features(ohlcv, RANDOM_STATE, use_hmm_extension=False)
full = merge_price_and_signal_data(features, signals_long)
full = add_signal_features(full)
features.shape, full.shape

## 5. Triple-Barrier Label Construction

In [ ]:
labels = triple_barrier_labels(
    ohlcv=ohlcv,
    signals_long=signals_long,
    feature_frame=features,
    horizon=DEFAULT_VERTICAL_BARRIER_DAYS,
    pt_mult=DEFAULT_PROFIT_TAKING_MULTIPLIER,
    sl_mult=DEFAULT_STOP_LOSS_MULTIPLIER,
)
validate_barrier_end_dates(labels, prediction_start=DEFAULT_PREDICTION_START)
label_diagnostics(labels, signals_long)

## 6. Model Training

In [ ]:
full = full.merge(labels, on=["date", "instrument"], how="left")
full = add_controlled_features(full)
spec = ExperimentSpec("signal_history_logistic", ("side_adjusted", "signal_history"), 2)
feature_cols = feature_columns_for_experiment(full, spec, use_hmm_extension=False)
validate_no_lookahead_features(full, feature_cols)
models = fit_fixed_family_final_models(
    experiment_name=spec.name,
    family="logistic",
    full=full,
    feature_cols=feature_cols,
    prediction_start=pd.Timestamp(DEFAULT_PREDICTION_START),
    seed=RANDOM_STATE,
    min_train_events=DEFAULT_MIN_TRAIN_EVENTS,
)
len(models), feature_cols[:10]

## 7. Evaluation

In [ ]:
# The full production run writes evaluation_summary.csv. This lightweight notebook cell shows
# where the public-window prediction frame is created without overwriting the frozen final file.
predictions = make_predictions(full, models, pd.Timestamp(DEFAULT_PREDICTION_START), pd.Timestamp(DEFAULT_PREDICTION_END))
validate_prediction_file(predictions)
predictions.head()

## 8. Feature Importance

In [ ]:
importance = cluster_permutation_importance(
    full_frame=full,
    models=models,
    prediction_start=pd.Timestamp(DEFAULT_PREDICTION_START),
    prediction_end=pd.Timestamp(DEFAULT_PREDICTION_END),
    seed=RANDOM_STATE,
)
importance.head()

## 9. Prediction Export

In [ ]:
# Safeguard: this notebook writes a candidate file by default.
out_path = DEFAULT_OUTPUT_DIR / "notebook_01_predictions_candidate.csv"
predictions.to_csv(out_path, index=False)
out_path

## 10. Final Audit

In [ ]:
from coursework.src.utils import sha256_file
sha256_file(out_path)